### Imports

In [75]:
from pathlib import Path
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

PyTorch version : 2.14.0+cu126
CUDA available  : True


### Project Paths

In [76]:
current_directory = Path.cwd()
tokenizer_directory = current_directory / "tokenizer"
tokenizer_path = tokenizer_directory / "tokenizer.json"
tokenizer_config_path = tokenizer_directory / "tokenizer_config.json"
print("Current directory :", current_directory)
print("Tokenizer exists  :", tokenizer_path.exists())
print("Config exists     :", tokenizer_config_path.exists())

Current directory : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model
Tokenizer exists  : True
Config exists     : True


In [77]:
## Load tokenizer
with open(tokenizer_config_path, 'r', encoding='utf-8') as file:
    tokenizer_config = json.load(file)

vocab_size = tokenizer_config['vocab_size']
print("Vocabulary size :", vocab_size)
print("Algorithm       :", tokenizer_config["algorithm"])

Vocabulary size : 21000
Algorithm       : WordPiece


### Model Configuration

In [78]:
model_config = {
    "vocab_size": vocab_size,          # the vocabulary size that our tokenizer outputs
    "max_sequence_length": 256,        # also called "context window", the lenght of input sequence in one go

    "hidden_size": 128,                # the dimensionality of each token's representation throughout the Transfomer Backbone : [B, 256] to [B, 256, 218]
    "num_hidden_layers": 4,            # number of transformer encoder blocks

    "num_attention_heads": 4,          # multi head attention - 4 different perspectives
    "head_dimension": 32,              # 128 /4 = 32

    "intermediate_size": 256,          # expansion size - The purpose is to give the model a larger intermediate space in which to perfom nonlinear transformations

    "hidden_dropout": 0.05,            # randomly neurons will be switched off in both the FFN layer and Attention layer
    "attention_dropout": 0.05,

    "layer_norm_eps": 1e-5,            # the epsilon in the denominator

    "rope_theta": 10000.0,             # base for rotation of embedding pairs 
    "rotary_dimension": 32,            # all positions of the Q,K matrices will rotate

    "initializer_std": 0.02,           # random weight initializer
}

for key, value in model_config.items():
    print(f"{key:24s}: {value}")

vocab_size              : 21000
max_sequence_length     : 256
hidden_size             : 128
num_hidden_layers       : 4
num_attention_heads     : 4
head_dimension          : 32
intermediate_size       : 256
hidden_dropout          : 0.05
attention_dropout       : 0.05
layer_norm_eps          : 1e-05
rope_theta              : 10000.0
rotary_dimension        : 32
initializer_std         : 0.02


In [79]:
assert (
    model_config["hidden_size"]
    == model_config["num_attention_heads"]
    * model_config["head_dimension"]
)

assert model_config["rotary_dimension"] <= model_config["head_dimension"]

assert model_config["rotary_dimension"] % 2 == 0

print("Model configuration is valid.")

Model configuration is valid.


### Tokenization 

In [80]:
## load tokenizer
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(str(tokenizer_path))           # loads a previously saved tokenizer, including its vocabulary, WordPiece model, normalization, pre-tokenization and other configurations
print("Tokenizer loaded.")
print("Vocabulary size :", tokenizer.get_vocab_size())

Tokenizer loaded.
Vocabulary size : 21000


In [81]:
## check tokenizer
sample_text = "The patient has patellofemoral chondromalacia."

encoding = tokenizer.encode(sample_text)

print("Text   :", sample_text)
print("Tokens :", encoding.tokens)
print("IDs    :", encoding.ids)

Text   : The patient has patellofemoral chondromalacia.
Tokens : ['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']
IDs    : [1970, 14284, 2186, 3675, 6781, 11885, 20391, 1936, 5521, 1930, 2003, 2804, 8046, 18]


### Embedding Layer

In [82]:
class TokenEmbedding(nn.Module):                                 # oops inheritance at work
    def __init__(self, vocab_size, hidden_size, dropout):        # defines what the layer contains
        super().__init__()

        self.token_embedding = nn.Embedding(                     # creates a learnable embedding matrix for each token in the vocab
            num_embeddings=vocab_size,
            embedding_dim=hidden_size                    
        )

        self.dropout = nn.Dropout(dropout)                       # creates a dropout layer - acts as regularization and prevents overfitting

    def forward(self, input_ids):

        embeddings = self.token_embedding(input_ids)

        embeddings = self.dropout(embeddings)

        return embeddings

In [83]:
## test token embeddings
input_ids = torch.tensor(                       # need to convert the ids to tensor
    [encoding.ids],                         # from the test run above
    dtype = torch.long
)

token_embedding = TokenEmbedding(                      # initializing our embedding object with the configurations we had set earlier
    vocab_size = model_config['vocab_size'],
    hidden_size =  model_config['hidden_size'],
    dropout = model_config['hidden_dropout']
)

embedding_output = token_embedding(input_ids)          # passing the input through the object

print("Tokens           :", encoding.tokens)
print("Input IDs shape  :", input_ids.shape)
print("Embedding shape  :", embedding_output.shape)

Tokens           : ['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']
Input IDs shape  : torch.Size([1, 14])
Embedding shape  : torch.Size([1, 14, 128])


In [84]:
torch.tensor(encoding.ids)

tensor([ 1970, 14284,  2186,  3675,  6781, 11885, 20391,  1936,  5521,  1930,
         2003,  2804,  8046,    18])

# Transformer Block

### Pre - Layer Norm

In [85]:
pre_attention_layer_norm = nn.LayerNorm(                    # pytorch's layernorm module. Layer norm acts on each token separately
    normalized_shape = model_config['hidden_size'],
    eps = model_config['layer_norm_eps']
)

normalized_embeddings = pre_attention_layer_norm(           # passing our embedded input through the layernorm
    embedding_output
)

print("Input shape          :", embedding_output.shape)
print("Normalized shape     :", normalized_embeddings.shape) # confirming if shapes remain same or not
                                                             # for every token, normalizes across its 128 hidden dimensions

Input shape          : torch.Size([1, 14, 128])
Normalized shape     : torch.Size([1, 14, 128])


### Query (Q) - Key (K) - Value(V) vectors

In [86]:
query_projection = nn.Linear(
    in_features = model_config['hidden_size'],
    out_features = model_config['hidden_size'],
    bias = False
)

key_projection = nn.Linear(
    in_features = model_config['hidden_size'],
    out_features = model_config['hidden_size'],
    bias = False
)

value_projection = nn.Linear(
    in_features = model_config['hidden_size'],
    out_features = model_config['hidden_size'],
    bias = False
)

query = query_projection(normalized_embeddings)
key = key_projection(normalized_embeddings)
value = value_projection(normalized_embeddings)

print("Input shape :", normalized_embeddings.shape)
print("Query shape :", query.shape)
print("Key shape   :", key.shape)
print("Value shape :", value.shape)

Input shape : torch.Size([1, 14, 128])
Query shape : torch.Size([1, 14, 128])
Key shape   : torch.Size([1, 14, 128])
Value shape : torch.Size([1, 14, 128])


In [87]:
## ^^ Mathematically ----- Q = XW_q, K = XW_k, V = XW_v

In [88]:
## inspect projection weights
print("Query weight shape :", query_projection.weight.shape)
print("Key weight shape   :", key_projection.weight.shape)
print("Value weight shape :", value_projection.weight.shape)

print("\nQuery bias :", query_projection.bias)
print("Key bias   :", key_projection.bias)
print("Value bias :", value_projection.bias)

Query weight shape : torch.Size([128, 128])
Key weight shape   : torch.Size([128, 128])
Value weight shape : torch.Size([128, 128])

Query bias : None
Key bias   : None
Value bias : None


### Multi Head split

In [89]:
## this cell will reshape our Q, K, V tensors into multiple attention heads

batch_size, sequence_length, hidden_size = query.shape

num_heads = model_config['num_attention_heads']
head_dimension = model_config['head_dimension']

query = query.view(                                # reshapes query from 3 dimensions to 4 dimensions
    batch_size, 
    sequence_length,
    num_heads,                                     # this is the new dimension we are adding, essentially inserting 4 matrices instead of the 1 currently
    head_dimension                                 # instead of hidden_size, now each head will have dimension hidden_size/4
)

key = key.view( 
    batch_size,
    sequence_length,
    num_heads,
    head_dimension
)

value = value.view(
    batch_size,
    sequence_length,
    num_heads,
    head_dimension
)

print("Query shape :", query.shape)
print("Key shape   :", key.shape)
print("Value shape :", value.shape)

Query shape : torch.Size([1, 14, 4, 32])
Key shape   : torch.Size([1, 14, 4, 32])
Value shape : torch.Size([1, 14, 4, 32])


In [90]:
## transpose Q,K,V into attention-head format

query = query.transpose(1,2)                     # switch 1st and 2nd dimensions
key = key.transpose(1,2)
value = value.transpose(1,2)

print("Query shape     :", query.shape)          # this arrangement allows each head to compute it's own attentions
print("Key shape       :", key.shape)            # Q*K^t = [1,4,14,32] * [1,4,32,14] = [1,4,14,14] >>> 14 x 14 attention matrix across each head
print("Value shape     :", value.shape)          # and then finally attention*value = [1,4,14,32] across all 4 heads >> concat to get [1,14,128]

Query shape     : torch.Size([1, 4, 14, 32])
Key shape       : torch.Size([1, 4, 14, 32])
Value shape     : torch.Size([1, 4, 14, 32])


### RoPe - We need to add the positional information as well

In [91]:
## because we each head has 32 dimensions, RoPE will have 16 pairs of dimensions in each head according to position
## Calculte RoPE frequencies
rotary_dimension = model_config['rotary_dimension']      
rope_theta = model_config['rope_theta']                  # the base constant of our rotation angle, controlling the frequency distribution

inverse_frequencies = 1.0 / (                            # 1/10000^^2i/d
    rope_theta ** (
        torch.arange(0, rotary_dimension, 2).float()     # generates even numbered dimension indices 0,2,4,6,...
        / rotary_dimension                               # divide by dimension value 
    )
)

print("Inverse frequencies shape :", inverse_frequencies.shape)
print("Inverse frequencies       :", inverse_frequencies)

Inverse frequencies shape : torch.Size([16])
Inverse frequencies       : tensor([1.0000e+00, 5.6234e-01, 3.1623e-01, 1.7783e-01, 1.0000e-01, 5.6234e-02,
        3.1623e-02, 1.7783e-02, 1.0000e-02, 5.6234e-03, 3.1623e-03, 1.7783e-03,
        1.0000e-03, 5.6234e-04, 3.1623e-04, 1.7783e-04])


In [92]:
## calculate the rotation angle 
sequence_length = query.shape[2]                  # the number of tokens

positions = torch.arange(                         # pytorch function that returns a 1D tensor containing seq of evenly spaced values (start, end, step)
    sequence_length,                              # we are defining the absolute token position
    dtype = torch.float32
)

angles = torch.outer(                             # python function to calculate the outer product of two 1D input vectors. N x M = [N,M]
    positions,                                    # 14
    inverse_frequencies                           # 16, leading to (14,16) matrix
)

cos_angles = torch.cos(angles)                    # recall rotation matrix  [cos(angle)    sin(angle)]
sin_angles = torch.sin(angles)                    #                         [-sin(angle)   cos(angle)]

print("Positions shape :", positions.shape)
print("Angles shape    :", angles.shape)
print("Cosine shape    :", cos_angles.shape)
print("Sine shape      :", sin_angles.shape)      # each row represents a token position and each column represents one of 16 rotation frequencies

Positions shape : torch.Size([14])
Angles shape    : torch.Size([14, 16])
Cosine shape    : torch.Size([14, 16])
Sine shape      : torch.Size([14, 16])


In [93]:
## Implement RoPE rotation

def apply_rope(x, cos_angles, sin_angles):                # x: [batch, heads, sequence, head_dimension]
    x_even = x[..., 0::2]                                 # extracts all even indexed dimensions 0,2,4,6
    x_odd = x[...,1::2]                                   # extracts all odd indexed dimensions 1,3,5,7

    cos = cos_angles[None, None, :, :]                    # adds two dimensions so cosine values can be broadcast across batches and attention heads
    sin = sin_angles[None, None, :, :]                    # same ^^   

    rotated_even = (                                      # computes the first coordinate of each rotated pair
        x_even * cos
        - x_odd * sin
    )

    rotated_odd = (                                       # computes the second coordinate of each rotated pair
        x_even * sin
        + x_odd * cos
    )

    rotated = torch.stack(                                # combines the rotated even and off values into pairs
        [rotated_even, rotated_odd],
        dim=-1
    )

    return rotated.flatten(start_dim=-2)                  # flatten the final two dimensions, restoring the original embedding layout

                                                          # original x :                  [2,4,16,32]
                                                          # x_even, x_odd :               [2,4,16,16]
                                                          # cos, sin :                    [1,1,16,16]
                                                          # rotated_even, rotated_odd :   [2,4,16,16]
                                                          # after torch.stack :           [2,4,16,16,16]
                                                          # flatten(dim=-2) :             [2,4,16,32]

In [94]:
## ^^ the above implementation assumes the RoPE is applied to the entire attention head ie. 32 dimensions
## RoPE is applied to Q and K vectors

In [95]:
## Apply RoPE to Q -- K vectors

query_rotated = apply_rope(query, cos_angles, sin_angles)      # both cos and sin angles have been computed above
key_rotated = apply_rope(key, cos_angles, sin_angles)

print("Original query shape :", query.shape)
print("Rotated query shape  :", query_rotated.shape)

print("Original key shape   :", key.shape)
print("Rotated key shape    :", key_rotated.shape)

print("Value shape          :", value.shape)

Original query shape : torch.Size([1, 4, 14, 32])
Rotated query shape  : torch.Size([1, 4, 14, 32])
Original key shape   : torch.Size([1, 4, 14, 32])
Rotated key shape    : torch.Size([1, 4, 14, 32])
Value shape          : torch.Size([1, 4, 14, 32])


In [96]:
## RoPE changes the orientation but preserves the Norm of the vectors - verification below

original_norms = torch.linalg.vector_norm(      # this function computes the vector norm of an input tensor (Eculidean or L2 norm)
    query,
    dim=-1
)

rotated_norms = torch.linalg.vector_norm(       # basically the length of a vector
    query_rotated,
    dim=-1
)

print(
    "Maximum norm difference:",
    torch.max(
        torch.abs(original_norms - rotated_norms)
    ).item()
)

print(
    "Norms preserved:",
    torch.allclose(
        original_norms,
        rotated_norms,
        atol=1e-6
    )
)

Maximum norm difference: 4.76837158203125e-07
Norms preserved: True


### Scaled Dot-Product Attention

In [97]:
head_dimension = model_config['head_dimension']       # already saved as a variable above but, just to be sure, we are explicitly calling it out again

attention_scores = torch.matmul(                      # torch' function for matrix multiplication
    query_rotated,                                    # our query vector post application of RoPE
    key_rotated.transpose(-2,-1)                       # our rotated key vector, which has been transposed
                                                      # [1,4,14,32] to [1,4,32,14]
)

attention_scores = attention_scores / math.sqrt(      # applying the dimension scaling
    head_dimension
)

print("Query shape            :", query_rotated.shape)
print("Key shape              :", key_rotated.shape)
print("Attention scores shape :", attention_scores.shape)
print("Scaling factor         :", math.sqrt(head_dimension))

Query shape            : torch.Size([1, 4, 14, 32])
Key shape              : torch.Size([1, 4, 14, 32])
Attention scores shape : torch.Size([1, 4, 14, 14])
Scaling factor         : 5.656854249492381


In [98]:
key_rotated.shape

torch.Size([1, 4, 14, 32])

In [99]:
key_rotated.transpose(-2,-1).shape

torch.Size([1, 4, 32, 14])

In [100]:
## inspect attention scores

first_head_scores = attention_scores[0, 0]                   # extracts attention scores for the first attention head of the first batch element
                                                             # typically [batch_size, num_heads, sequence_length, sequence_length]
print("Tokens:")
print(encoding.tokens)

print("\nFirst attention head scores:")
print(first_head_scores.detach().round(decimals=3))          # detach creates a tensor detached from PyTorch's computational graph, so operations on it are not tracked for gradient computation

print("\nScores for the first token:")                       # selects the first row of this attention matrix and again detaches it
print(first_head_scores[0].detach().round(decimals=3))

Tokens:
['The', 'patient', 'has', 'pat', '##ello', '##fe', '##mor', '##al', 'cho', '##nd', '##rom', '##ala', '##cia', '.']

First attention head scores:
tensor([[-1.3400e-01,  2.3800e-01,  5.6400e-01, -4.4000e-02,  1.7200e-01,
         -2.5500e-01, -2.9900e-01, -1.3100e-01,  1.8300e-01,  1.2400e-01,
         -1.7600e-01, -1.6900e-01,  5.8400e-01,  1.9300e-01],
        [-3.1800e-01,  9.7000e-02,  2.3300e-01,  8.5000e-02,  2.2300e-01,
         -1.8500e-01,  1.6900e-01,  8.2000e-02, -6.3000e-02, -1.7000e-01,
          8.3000e-02, -2.6800e-01,  5.7100e-01,  3.5200e-01],
        [-1.7700e-01,  1.9200e-01,  2.7500e-01, -2.1700e-01,  9.7000e-02,
         -3.7900e-01,  1.0500e-01, -4.3300e-01,  1.5900e-01,  4.6000e-02,
         -1.4900e-01, -1.5300e-01, -1.5400e-01, -2.3300e-01],
        [ 5.0700e-01, -1.0800e-01, -2.7900e-01, -2.1300e-01, -2.9100e-01,
          3.2500e-01,  1.5700e-01,  7.6100e-01, -4.0000e-02,  6.1000e-02,
          1.9300e-01,  3.4700e-01,  1.3300e-01, -2.0000e-03],
       

### Padding / Attention Mask

In [101]:
## creating an attention mask that distinguishes actual tokens from padding tokens 

pad_token_id = tokenizer.token_to_id("[PAD]")                # retrieves the integer ID assigned to [PAD] by our tokenizer

attention_mask = (
    input_ids != pad_token_id                                # check whether each token is a pad or not
).long()                                                     # returns true or false, long converts to 1 or 0
                                                             # Without a padding mask, the model could assign attention weight to meaningless [PAD] positions
                                                             # the mask tells the model which key positions should be ignored

print("PAD token ID      :", pad_token_id)
print("Input IDs         :", input_ids)
print("Attention mask    :", attention_mask)
print("Attention shape   :", attention_mask.shape)

PAD token ID      : 0
Input IDs         : tensor([[ 1970, 14284,  2186,  3675,  6781, 11885, 20391,  1936,  5521,  1930,
          2003,  2804,  8046,    18]])
Attention mask    : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Attention shape   : torch.Size([1, 14])


In [102]:
## apply padding mask to attention scores
expanded_attention_mask = attention_mask[:, None, None, :]           # attention mask originally has shape (batch_size, seq_len)
                                                                     # None means insert a new dimension of size 1 - add dimensions for attention heads and query positions

masked_attention_scores = attention_scores.masked_fill(              # (condition, value)
    expanded_attention_mask == 0,                                    # identifies padding positions
    torch.finfo(attention_scores.dtype).min                          # retireves the numerical data type of the attention score
                                                                     # returns the smallest finite representable value for that floating point type
)                                                                    # although shape is (2,1,1,16) - python broadcasting takes care of the application of this padding mask across all attention heads and query positions

print("Original scores shape :", attention_scores.shape)
print("Expanded mask shape   :", expanded_attention_mask.shape)
print("Masked scores shape   :", masked_attention_scores.shape)

Original scores shape : torch.Size([1, 4, 14, 14])
Expanded mask shape   : torch.Size([1, 1, 1, 14])
Masked scores shape   : torch.Size([1, 4, 14, 14])


### Softmax to convert attention score into probs

In [103]:
attention_probabilities = F.softmax(
    masked_attention_scores,
    dim=-1
)

print("Attention probabilities shape :",attention_probabilities.shape)
print("\nFirst head, first token probabilities:")
print(attention_probabilities[0, 0, 0].detach().round(decimals=4))
print("\nSum of probabilities:",attention_probabilities[0, 0, 0].sum().item())

Attention probabilities shape : torch.Size([1, 4, 14, 14])

First head, first token probabilities:
tensor([0.0566, 0.0821, 0.1137, 0.0619, 0.0768, 0.0501, 0.0480, 0.0567, 0.0776,
        0.0732, 0.0542, 0.0546, 0.1160, 0.0785])

Sum of probabilities: 1.0000001192092896


In [104]:
## ^^ IMPORTANT - this is bi directional self attention, we are not using causal mask, 
## so token can attent to valid tokens before and after itself

### Attention Dropout

In [105]:
attention_dropout = nn.Dropout(
    p=model_config["attention_dropout"]
)

attention_weights = attention_dropout(
    attention_probabilities
)

print("Attention probabilities shape :", attention_probabilities.shape)
print("Attention weights shape       :", attention_weights.shape)

Attention probabilities shape : torch.Size([1, 4, 14, 14])
Attention weights shape       : torch.Size([1, 4, 14, 14])


In [106]:
## ^^ Important: Dropout is active because the module is in training mode. During inference, we'll use .eval() to disable it.

In [107]:
## Compute attention weight values
attention_output = torch.matmul(                                   # Softmax(attention scores) * Value vector
    attention_weights,
    value
)

print("Attention weights shape :", attention_weights.shape)
print("Value shape             :", value.shape)
print("Attention output shape  :", attention_output.shape)

Attention weights shape : torch.Size([1, 4, 14, 14])
Value shape             : torch.Size([1, 4, 14, 32])
Attention output shape  : torch.Size([1, 4, 14, 32])


In [108]:
## Concatenating Attention heads
## We have four heads, each producing 32-dimensional representations
## We now combine them into a single 128-dimensional representation per token

attention_output = attention_output.transpose(1, 2)               # Swaps dimensions 1 and 2

attention_output = attention_output.reshape(                      # changes the tensor's shape while preserving it's values
    batch_size,                                                   
    sequence_length,
    model_config["hidden_size"]                                   # combines num_head and head_dimension into a single dim
                                                                  # works because hidden_size = num_heads * head_dim
)

print("Concatenated attention output shape :", attention_output.shape)

Concatenated attention output shape : torch.Size([1, 14, 128])


### Output Linear Projection

In [109]:
## this projection completes the multi-head attention

In [110]:
output_projection = nn.Linear(
    in_features=model_config["hidden_size"],
    out_features=model_config["hidden_size"]
)

projected_attention_output = output_projection(
    attention_output
)

print("Input shape      :", attention_output.shape)
print("Projection weight:", output_projection.weight.shape)
print("Output shape     :", projected_attention_output.shape)

Input shape      : torch.Size([1, 14, 128])
Projection weight: torch.Size([128, 128])
Output shape     : torch.Size([1, 14, 128])


### Residual connection and LayerNorm

In [111]:
attention_dropout = nn.Dropout(p=model_config['hidden_dropout'])     # notice now we are using hidden dropout and not attention, because this is the output from the linear projection
attention_layer_norm = nn.LayerNorm(model_config['hidden_size'])     # once again layernorm will be applied

attention_output_dropped = attention_dropout(projected_attention_output)
residual_output = (                                                              # Add original input (residual connection)
    embedding_output + attention_output_dropped
)

normalized_attention_output = attention_layer_norm(
    residual_output                                                              # apply layer norm to the combined output
)

print("Embedding shape   :", embedding_output.shape)
print("Attention shape   :", attention_output_dropped.shape)
print("Residual shape    :", residual_output.shape)
print("Normalized shape  :", normalized_attention_output.shape)

Embedding shape   : torch.Size([1, 14, 128])
Attention shape   : torch.Size([1, 14, 128])
Residual shape    : torch.Size([1, 14, 128])
Normalized shape  : torch.Size([1, 14, 128])


In [112]:
## attention sub-layer is complete with the above residual connection

### Feed Forward Network of the TRANSFORMER BLOCK

In [114]:
## The FFN works independently on each token, unlike self-attention, which exchanges information between tokens

In [117]:
## define SwiGLU projections
ffn_gate = nn.Linear(
    in_features=model_config["hidden_size"],
    out_features=model_config["intermediate_size"],
    bias=False
)

ffn_value = nn.Linear(
    in_features=model_config["hidden_size"],
    out_features=model_config["intermediate_size"],
    bias=False
)

ffn_output = nn.Linear(
    in_features=model_config["intermediate_size"],
    out_features=model_config["hidden_size"],
    bias=False
)

print("Gate weight shape   :", ffn_gate.weight.shape)
print("Value weight shape  :", ffn_value.weight.shape)
print("Output weight shape :", ffn_output.weight.shape)

Gate weight shape   : torch.Size([256, 128])
Value weight shape  : torch.Size([256, 128])
Output weight shape : torch.Size([128, 256])


In [ ]:
## preLN 
pre_ffn_layer_norm = nn.LayerNorm(                          # the input in this block goes through the LayerNorm
    normalized_shape=model_config["hidden_size"],
    eps=model_config["layer_norm_eps"]
)